In [ ]:
# 挂载Drive + 安装依赖
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

import os
import numpy as np
from PIL import Image
import shutil
print('✅ 环境准备完成')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.8 MB/s eta 0:00:00
✅ 环境准备完成


In [ ]:
# 将VOC格式数据集转换为YOLO分割格式
import os
import numpy as np
from PIL import Image
import shutil

def voc_mask_to_yolo_seg(mask_path, img_w, img_h):
    """
    将语义分割mask转换为YOLO polygon格式
    返回: list of "class_id x1 y1 x2 y2 ..." 字符串
    """
    import cv2
    mask = np.array(Image.open(mask_path).convert('L'))
    # 二值化：>0为植被(class 0，YOLO只有植被一类)
    binary = (mask > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 50:  # 过滤极小噪点
            continue
        cnt = cnt.squeeze()
        if cnt.ndim < 2 or len(cnt) < 3:
            continue
        # 归一化坐标
        coords = []
        for pt in cnt:
            coords.append(f'{pt[0]/img_w:.6f} {pt[1]/img_h:.6f}')
        lines.append('0 ' + ' '.join(coords))  # class_id=0 表示植被
    return lines


def convert_dataset(data_dir, output_dir, split='train'):
    """将单年数据转换并保存到output_dir"""
    img_src  = os.path.join(data_dir, 'JPEGImages')
    mask_src = os.path.join(data_dir, 'SegmentationClass')

    img_dst   = os.path.join(output_dir, 'images', split)
    label_dst = os.path.join(output_dir, 'labels', split)
    os.makedirs(img_dst,   exist_ok=True)
    os.makedirs(label_dst, exist_ok=True)

    fnames = sorted([f for f in os.listdir(img_src)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    success, skip = 0, 0
    for fname in fnames:
        stem      = os.path.splitext(fname)[0]
        img_path  = os.path.join(img_src,  fname)
        mask_path = os.path.join(mask_src, stem + '.png')
        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_src, stem + '.jpg')
        if not os.path.exists(mask_path):
            skip += 1
            continue

        img = Image.open(img_path)
        w, h = img.size

        # 复制图像（统一转为jpg）
        dst_img = os.path.join(img_dst, stem + '.jpg')
        img.convert('RGB').save(dst_img, 'JPEG')

        # 生成YOLO label
        lines = voc_mask_to_yolo_seg(mask_path, w, h)
        with open(os.path.join(label_dst, stem + '.txt'), 'w') as f:
            f.write('\n'.join(lines))
        success += 1

    print(f'  [{split}] 转换完成: {success} 张，跳过: {skip} 张')


# ---- 执行转换 ----
YOLO_DATA = '/content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/dataset'

print('🔄 转换2022数据 → train...')
convert_dataset('/content/drive/MyDrive/datasets/2022-seg', YOLO_DATA, split='train')

print('🔄 转换2023数据 → train（追加）...')
convert_dataset('/content/drive/MyDrive/datasets/2023-seg', YOLO_DATA, split='train')

print('🔄 转换2024数据 → val/test...')
convert_dataset('/content/drive/MyDrive/datasets/2024-seg', YOLO_DATA, split='val')

# 统计
train_imgs = len(os.listdir(os.path.join(YOLO_DATA, 'images', 'train')))
val_imgs   = len(os.listdir(os.path.join(YOLO_DATA, 'images', 'val')))
print(f'\n✅ 数据集转换完成！训练: {train_imgs} 张 | 验证: {val_imgs} 张')

🔄 转换2022数据 → train...
  [train] 转换完成: 67 张，跳过: 0 张
🔄 转换2023数据 → train（追加）...
  [train] 转换完成: 67 张，跳过: 0 张
🔄 转换2024数据 → val/test...
  [val] 转换完成: 67 张，跳过: 0 张

✅ 数据集转换完成！训练: 134 张 | 验证: 67 张


In [3]:
# 生成YOLO数据集配置文件
yaml_content = f"""
path: {YOLO_DATA}
train: images/train
val:   images/val

nc: 1
names:
  0: vegetation
"""

yaml_path = os.path.join(YOLO_DATA, 'vegetation.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print(f'✅ 数据集配置文件已生成: {yaml_path}')
print(yaml_content)

✅ 数据集配置文件已生成: /content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/dataset/vegetation.yaml

path: /content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/dataset
train: images/train
val:   images/val

nc: 1
names:
  0: vegetation



In [4]:
# 训练YOLO11-seg
from ultralytics import YOLO

SAVE_DIR  = '/content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/checkpoints'
CKPT_PATH = '/content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/weights/yolo11s-seg.pt'

os.makedirs(SAVE_DIR, exist_ok=True)

print('📦 加载YOLO11s-seg...')
model = YOLO(CKPT_PATH)

print('🚀 开始训练...')
results = model.train(
    data      = yaml_path,
    epochs    = 100,
    imgsz     = 512,        # YOLO支持任意尺寸，用512与原始数据一致
    batch     = 8,
    device    = 'cuda',
    optimizer = 'AdamW',
    lr0       = 5e-4,       # 初始学习率
    lrf       = 0.01,       # 最终学习率 = lr0 * lrf
    momentum  = 0.937,
    weight_decay = 1e-4,
    warmup_epochs = 3,
    cos_lr    = True,       # cosine学习率调度
    augment   = True,       # 内置数据增强
    hsv_h     = 0.015,      # 色调增强
    hsv_s     = 0.4,        # 饱和度增强
    hsv_v     = 0.3,        # 亮度增强
    flipud    = 0.5,        # 垂直翻转
    fliplr    = 0.5,        # 水平翻转
    mosaic    = 0.5,        # mosaic增强（小样本有效）
    project   = SAVE_DIR,
    name      = 'yolo11s_vegetation',
    save      = True,
    save_period = 10,       # 每10轮保存一次
    val       = True,
    plots     = True,       # 自动生成训练曲线图
    verbose   = True,
)
print('✅ 训练完成！')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 加载YOLO11s-seg...
🚀 开始训练...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/dataset/vegetation.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, 

In [5]:
# 测试集评估
from ultralytics import YOLO
import json

# 加载最优模型
best_ckpt = os.path.join(SAVE_DIR, 'yolo11s_vegetation', 'weights', 'best.pt')
model_best = YOLO(best_ckpt)

print('📊 在2024测试集上评估...')
metrics = model_best.val(
    data   = yaml_path,
    split  = 'val',
    imgsz  = 512,
    batch  = 8,
    device = 'cuda',
    plots  = True,
)

print(f'\n📊 2024测试集最终结果:')
print(f'   mIoU (mask):    {metrics.seg.map:.4f}')
print(f'   mIoU@50:        {metrics.seg.map50:.4f}')
print(f'   mIoU@75:        {metrics.seg.map75:.4f}')
print(f'   Precision:      {metrics.seg.mp:.4f}')
print(f'   Recall:         {metrics.seg.mr:.4f}')

# 保存结果
result_dict = {
    'mIoU':      float(metrics.seg.map),
    'mIoU@50':   float(metrics.seg.map50),
    'mIoU@75':   float(metrics.seg.map75),
    'Precision': float(metrics.seg.mp),
    'Recall':    float(metrics.seg.mr),
}
with open(os.path.join(SAVE_DIR, 'yolo11s_test_results.json'), 'w') as f:
    json.dump(result_dict, f, indent=2)
print('✅ 测试结果已保存')

📊 在2024测试集上评估...
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,203 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 65.4±11.2 MB/s, size: 88.9 KB)
val: Scanning /content/drive/MyDrive/vegetation_models_v2/3_YOLO11seg/dataset/labels/val.cache... 67 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 67/67 31.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.5it/s 2.6s
                   all         67       1646      0.338      0.284      0.231      0.103      0.309      0.229      0.182      0.067
Speed: 1.7ms preprocess, 13.0ms inference, 0.0ms loss, 5.2ms postprocess per image
Results saved to /content/runs/segment/val

📊 2024测试集最终结果:
   mIoU (mask):    0.0670
   mIoU@50:        0.1822
   mIoU@75:        0.0442
   Precision:      0.

In [6]:
# 可视化预测结果
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

val_img_dir = os.path.join(YOLO_DATA, 'images', 'val')
val_imgs    = sorted(os.listdir(val_img_dir))[:6]

fig, axes = plt.subplots(6, 2, figsize=(10, 24))
axes[0][0].set_title('原图', fontsize=13)
axes[0][1].set_title('YOLO11预测', fontsize=13)

for i, fname in enumerate(val_imgs):
    img_path = os.path.join(val_img_dir, fname)
    img_bgr  = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 推理
    result = model_best.predict(img_path, imgsz=512,
                                 conf=0.25, device='cuda', verbose=False)[0]

    # 绘制mask
    pred_img = img_rgb.copy()
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        for m in masks:
            m_resized = cv2.resize(m, (img_rgb.shape[1], img_rgb.shape[0]))
            pred_img[m_resized > 0.5] = [0, 200, 0]  # 绿色覆盖植被

    axes[i][0].imshow(img_rgb)
    axes[i][0].set_ylabel(fname, fontsize=7, rotation=0, labelpad=70)
    axes[i][1].imshow(pred_img)
    for ax in axes[i]: ax.axis('off')

fig.legend(handles=[
    mpatches.Patch(color='green', label='植被预测'),
], loc='lower center', fontsize=12, bbox_to_anchor=(0.5, 0.01))
plt.suptitle(f'YOLO11s-seg 2024测试集预测结果\n'
             f'mIoU={result_dict["mIoU"]:.4f} | '
             f'P={result_dict["Precision"]:.4f} | '
             f'R={result_dict["Recall"]:.4f}',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'yolo11s_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ 可视化已保存')

Output hidden; open in https://colab.research.google.com to view.